# Structured Information Extraction with Pydantic


In [ ]:
def TODO(todo: str = "Fill the blank"):
    raise ValueError(todo)

In [49]:
import os
from openai import OpenAI
import pydantic
import random
from PIL import Image
import json

from ie_course.image import encode_image

In [50]:
img_bytes = encode_image("../../data/gemini_generated_invoice.png")

key = os.environ[TODO("Environment variable containing the API key")]
model = TODO("Model name")
client = OpenAI(api_key=key, base_url="https://inference.dev.ellipsis-drive.com/v1")

## Define a schema with Pydantic

Instead of describing the shape in prose, we declare it once as a Pydantic model. This single
artifact is the **schema** we send to the server, the **validator** we run on the reply, and the
**typed object** we ultimately work with.

Note the things doing real work:
- `Field(description=...)` — sent to the model as part of the schema. Free prompting.
- `Enum` — constrains the *value*, not just the type.
- a nested model (`LineItem`) inside a `list` — repeated entities.
- `Optional[...] = None` — strict schemas have no "missing"; optional means "may be null".

In [51]:
from enum import Enum
from typing import Optional
from datetime import date
from pydantic import BaseModel, Field

class InvoiceStatus(str, Enum):
    paid = "paid"
    unpaid = "unpaid"
    partially_paid = "partially_paid"

class LineItem(BaseModel):
    description: str = Field(description="Name/description of the purchased item")
    quantity: float = Field(description="Number of units; integer")
    unit_price: float = Field(description="Price per single unit, in the invoice currency")

class Invoice(BaseModel):
    vendor_name: str = Field(description="The company that issued the invoice")
    invoice_number: str = Field(description="The invoice identifier, e.g. INV-2024-0915")
    issue_date: date = Field(description="Date the invoice was issued (ISO 8601)")
    currency: str = Field(description="3-letter ISO currency code, e.g. USD")
    status: InvoiceStatus = Field(description="Payment status")
    line_items: list[LineItem] = Field(description="The individual billed items")
    total_amount: float = Field(description="The total amount due")
    purchase_order: Optional[str] = Field(default=None, description="PO number if present, else null")

print("Schema defined. Fields:", list(Invoice.model_fields))

Schema defined. Fields: ['vendor_name', 'invoice_number', 'issue_date', 'currency', 'status', 'line_items', 'total_amount', 'purchase_order']


## Inspect the schema the server receives

`chat_json` sent `Invoice.model_json_schema()`. Let's inspect it and connect it to the strict-mode
rules from the lecture. Even when a server only *loosely* honors the schema, these are the
constraints a constrained-decoding backend (e.g. vLLM `guided_json`) compiles into a grammar.

In [52]:
schema = TODO("Create the JSON schema from the Invoice model")
print(json.dumps(schema, indent=2))

{
  "$defs": {
    "InvoiceStatus": {
      "enum": [
        "paid",
        "unpaid",
        "partially_paid"
      ],
      "title": "InvoiceStatus",
      "type": "string"
    },
    "LineItem": {
      "properties": {
        "description": {
          "description": "Name/description of the purchased item",
          "title": "Description",
          "type": "string"
        },
        "quantity": {
          "description": "Number of units; integer",
          "title": "Quantity",
          "type": "integer"
        },
        "unit_price": {
          "description": "Price per single unit, in the invoice currency",
          "title": "Unit Price",
          "type": "number"
        }
      },
      "required": [
        "description",
        "quantity",
        "unit_price"
      ],
      "title": "LineItem",
      "type": "object"
    }
  },
  "properties": {
    "vendor_name": {
      "description": "The company that issued the invoice",
      "title": "Vendor Name",
      

## Schema-guided extraction over the open-weight endpoint

Now the portable pattern. We hand the server our schema (via `chat_json`, which negotiates
`json_schema` → `guided_json` → prompt), then **validate the returned string ourselves** with
`Invoice.model_validate_json(...)`. That client-side validation is what actually guarantees we
end up with a well-typed `Invoice`, regardless of how much the server enforced.

In [1]:
def chat_json(messages, schema, max_tokens=16_000):
    resp = client.chat.completions.create(
        model=model,
        messages=messages,
        max_tokens=max_tokens,
        response_format={
            "type": "json_schema",
            "json_schema": {"name": "extraction", "schema": schema, "strict": True},
        },
        temperature=0
    )
    return TODO("Return the raw response string")

In [55]:
# Exercise: Complete the extraction function.
# Fill in the missing schema and/or message components.

from pydantic import ValidationError

def extract_invoice(img_bytes):
    schema = TODO("Invoice JSON schema")

    messages = TODO("Messages for the extraction request")

    raw = chat_json(messages, schema)
    data = json.loads(raw)
    return Invoice.model_validate(data)

In [56]:
print(raw)

{
  "vendor_name": "TechForward Solutions",
  "invoice_number": "INV-2024-03-15-A",
  "issue_date": "2024-03-15",
  "currency": "USD",
  "status": "unpaid",
  "line_items": [
    {
      "description": "Custom API Integration Services (5 hours)",
      "quantity": 5,
      "unit_price": 150.00
    },
    {
      "description": "Cloud Storage Setup (Annual License)",
      "quantity": 1,
      "unit_price": 120.00
    },
    {
      "description": "Data Migration Support (Phase 2)",
      "quantity": 2,
      "unit_price": 200.00
    },
    {
      "description": "Monthly Maintenance Fee (March)",
      "quantity": 1,
      "unit_price": 75.00
    },
    {
      "description": "Project Consultation Workshop (Remote)",
      "quantity": 1,
      "unit_price": 300.00
    }
  ],
  "total_amount": 1888.96,
  "purchase_order": null
}


# Exercise: Compute the line-item sum and compare it with invoice.total_amount.

In [2]:
computed = TODO("Sum over quantity * unit_price for all line items")
print("Computed:", computed)
print("Invoice total:", invoice.total_amount)

NameError: name 'TODO' is not defined

Find `purchase_order`: it's modeled as a union with `null` (from `Optional[str]`), which is why
in strict schemas you get an explicit `null` rather than a missing key. Find `InvoiceStatus`: its
allowed values are enumerated, so a constrained backend cannot emit `"refunded"`.

> **Note on portability.** OpenAI's strict mode additionally requires `additionalProperties:false`
> and every field in `required`. Some open-weight servers are stricter or looser about this. If a
> server rejects the schema, our `chat_json` already falls back — and Pydantic still validates.

## Tool / function calling (portable form)

Many open-weight endpoints also support **tool calling** via the standard `tools` parameter —
useful when the model must *choose among actions* (e.g. extract vs. flag-not-an-invoice) rather
than always return one object. Support varies by server and model, so we again validate the
arguments ourselves with Pydantic and fall back if tools aren't available.

In [64]:
def extract_via_tool(img_bytes):

    tools = [{
        "type": "function",
        "function": {
            "name": "save_invoice",
            "description": "Save the structured invoice extracted from the text.",
            "parameters": Invoice.model_json_schema(),
        },
    }]

    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "Extract the invoice by calling save_invoice."},
            {
                "role": "user", 
                "content": [
                    {
                        "type": "image_url",
                        "image_url": {"url": f"data:image/png;base64,{img_bytes}"}
                    },
                ]
            }
        ],
        tools=tools,
        tool_choice={"type": "function", "function": {"name": "save_invoice"}},
    )
    TODO("return the function arguments")


tool_params = extract_via_tool(img_bytes)
tool_invoice = Invoice.model_validate_json(tool_invoice)

In [65]:
print(tool_params)

{
  "vendor_name": "TechForward Solutions",
  "invoice_number": "INV-2024-03-15-A",
  "issue_date": "2024-03-15",
  "currency": "USD",
  "status": "unpaid",
  "line_items": [
    {
      "description": "Custom API Integration Services (5 hours)",
      "quantity": 5,
      "unit_price": 150.00
    },
    {
      "description": "Cloud Storage Setup (Annual License)",
      "quantity": 1,
      "unit_price": 120.00
    },
    {
      "description": "Data Migration Support (Phase 2)",
      "quantity": 2,
      "unit_price": 200.00
    },
    {
      "description": "Monthly Maintenance Fee (March)",
      "quantity": 1,
      "unit_price": 75.00
    },
    {
      "description": "Project Consultation Workshop (Remote)",
      "quantity": 1,
      "unit_price": 300.00
    }
  ],
  "total_amount": 1888.96,
  "purchase_order": null
}


### Parsing and validation are where things break

With the portable pattern, two distinct failures can occur on the client:
- **invalid JSON** — the reply wasn't valid JSON at all (worse on un-enforced servers).
- **`pydantic.ValidationError`** — valid JSON, but it doesn't match our schema (wrong type, bad enum).

`model_validate_json` raises `ValidationError` for both the malformed-JSON and schema-mismatch
cases, so a single `except` covers them.

In [ ]:
def safe_extract(text):
    schema = Invoice.model_json_schema()
    messages = [
        {"role": "system", "content": "Extract the invoice details as one JSON object. Use null for missing fields."},
        {"role": "user", "content": text},
    ]
    raw = chat_json(messages, schema=schema)
    try:
        return Invoice.model_validate_json(raw)
    except ValidationError as e:
        print("Validation failed on the model reply:")
        print(e)
        raise

print(safe_extract(SAMPLE_INVOICE_TEXT).vendor_name)

### Validators catch the *sporadic* semantic errors

Strict schemas guarantee types, not meaning. A `field_validator` is the right place to catch
errors the model makes occasionally — e.g. a negative total. **Caveat from the lecture:** if the
model gets something wrong *consistently*, retries fail consistently too.

In [75]:
from pydantic import field_validator

class ValidatedInvoice(Invoice):
    @field_validator("total_amount")
    @classmethod
    def total_must_be_positive(cls, v):
        if v < 0:
            raise ValueError("total_amount must be positive")
        return v

    @field_validator("currency")
    @classmethod
    def currency_is_three_letters(cls, v):
        if len(v) != 3 or not v.isalpha():
            raise ValueError("currency must be a 3-letter ISO code")
        return v.upper()

# Demonstrate the validator firing on bad data:
try:
    ValidatedInvoice.model_validate(invoice.model_dump())
except ValidationError as e:
    print("Validation failed:\n", e)